# Barrier Option Engine — Interactive Exploration

Explores the up-and-out call engine in `options_engine.pricing.barrier` and
`options_engine.greeks.barrier_greeks`. Every plotting function below takes
its inputs as plain arguments, so re-running a cell with different numbers
is enough on its own — the `interact(...)` wrapper on top just adds sliders
so you don't have to.

**Kernel:** select this repo's `.venv` interpreter (Kernel → Change Kernel).
`ipykernel` and `ipywidgets` were added as dev dependencies for this notebook.

In [ ]:
import sys
from pathlib import Path

# Make the repo importable regardless of where Jupyter's working directory ends up.
repo_root = Path.cwd()
while not (repo_root / "options_engine").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interact_manual, FloatSlider, IntSlider

from options_engine.pricing.black_scholes import black_scholes_price
from options_engine.pricing.barrier import barrier_price_analytic, barrier_price_mc, simulate_gbm_paths
from options_engine.greeks.barrier_greeks import barrier_delta, barrier_gamma

%matplotlib inline

## 1. Base case

Plain print, no widgets — the fastest way to sanity-check a specific set of
numbers matches what you'd expect before exploring around them below.

In [ ]:
spot, strike, barrier, time_to_expiry, rate, sigma = 90.0, 100.0, 130.0, 1.0, 0.03, 0.25
steps = 252

vanilla = black_scholes_price(spot, strike, time_to_expiry, rate, sigma, "call")
continuous = barrier_price_analytic(spot, strike, barrier, time_to_expiry, rate, sigma)
matched = barrier_price_analytic(spot, strike, barrier, time_to_expiry, rate, sigma, monitoring_steps=steps)
mc = barrier_price_mc(spot, strike, barrier, time_to_expiry, rate, sigma, sims=100_000, steps=steps, seed=1)

print(f"Vanilla call:                         {vanilla:.4f}")
print(f"Up-and-out (continuous, closed form):  {continuous:.4f}")
print(f"Up-and-out (matched to {steps} steps):    {matched:.4f}")
print(f"Up-and-out (Monte Carlo, {steps} steps):  {mc:.4f}")

## 2. Sample paths against the barrier

Teal paths survive to expiry; red paths touch or cross the barrier and are
extinguished on the spot. Drag any slider and the plot regenerates.

In [ ]:
def plot_paths(spot=90.0, strike=100.0, barrier=130.0, sigma=0.25, time_to_expiry=1.0, n_paths=60, seed=7):
    steps = 252
    paths = simulate_gbm_paths(spot, time_to_expiry, 0.03, sigma, sims=n_paths, steps=steps, seed=seed)
    full_paths = np.hstack([np.full((n_paths, 1), spot), paths])
    t_grid = np.linspace(0, time_to_expiry, steps + 1)
    breached = np.any(paths >= barrier, axis=1)

    fig, ax = plt.subplots(figsize=(9, 5))
    for i in range(n_paths):
        color = "#c0392b" if breached[i] else "#1f7a72"
        ax.plot(t_grid, full_paths[i], color=color, linewidth=1, alpha=0.7 if breached[i] else 0.45)

    ax.axhline(barrier, color="#b8862a", linestyle="--", linewidth=1.5, label=f"barrier = {barrier:.1f}")
    ax.axhline(strike, color="#555555", linestyle=":", linewidth=1, label=f"strike = {strike:.1f}")
    ax.set_xlabel("time (years)")
    ax.set_ylabel("spot price")
    survived = n_paths - int(breached.sum())
    ax.set_title(f"{survived}/{n_paths} paths survived to expiry without breaching")
    ax.legend()
    plt.show()


interact(
    plot_paths,
    spot=FloatSlider(value=90.0, min=60.0, max=125.0, step=1.0),
    strike=FloatSlider(value=100.0, min=60.0, max=140.0, step=1.0),
    barrier=FloatSlider(value=130.0, min=105.0, max=160.0, step=1.0),
    sigma=FloatSlider(value=0.25, min=0.05, max=0.6, step=0.01),
    time_to_expiry=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1),
    n_paths=IntSlider(value=60, min=10, max=200, step=10),
    seed=IntSlider(value=7, min=0, max=100, step=1),
);

## 3. Convergence: Monte Carlo → analytic

Left: MC price vs. simulation count, at fixed monitoring frequency — pure MC
noise shrinking. Right: MC price vs. monitoring frequency, with *both* the
true continuous-monitoring formula and the BGK-matched formula shown — this
is the discrete-monitoring bias from the build guide's step 6, closing as
`steps` grows.

This one reruns several Monte Carlo simulations per click, so it uses
**Run Interact** instead of live sliders — it'd lag if it recomputed on
every drag.

In [ ]:
def plot_convergence(spot=90.0, strike=100.0, barrier=130.0, sigma=0.25, time_to_expiry=1.0, rate=0.03):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # (a) price vs. sims, monitoring frequency fixed
    steps_fixed = 200
    target = barrier_price_analytic(spot, strike, barrier, time_to_expiry, rate, sigma, monitoring_steps=steps_fixed)
    sim_counts = [1_000, 3_000, 10_000, 30_000, 100_000, 300_000]
    mc_prices = [
        barrier_price_mc(spot, strike, barrier, time_to_expiry, rate, sigma, sims=s, steps=steps_fixed, seed=42)
        for s in sim_counts
    ]
    axes[0].plot(sim_counts, mc_prices, "o-", color="#1f7a72", label="Monte Carlo")
    axes[0].axhline(target, color="#b8862a", linestyle="--", label="analytic (matched)")
    axes[0].set_xscale("log")
    axes[0].set_xlabel("simulations")
    axes[0].set_ylabel("price")
    axes[0].set_title(f"Convergence vs. simulation count ({steps_fixed} steps)")
    axes[0].legend()

    # (b) price vs. monitoring steps, sims fixed -- the discrete-monitoring bias closing
    sims_fixed = 200_000
    step_counts = [10, 25, 50, 100, 252, 500]
    continuous_price = barrier_price_analytic(spot, strike, barrier, time_to_expiry, rate, sigma)
    matched_prices = [
        barrier_price_analytic(spot, strike, barrier, time_to_expiry, rate, sigma, monitoring_steps=s)
        for s in step_counts
    ]
    mc_prices_2 = [
        barrier_price_mc(spot, strike, barrier, time_to_expiry, rate, sigma, sims=sims_fixed, steps=s, seed=42)
        for s in step_counts
    ]
    axes[1].plot(step_counts, mc_prices_2, "o-", color="#1f7a72", label="Monte Carlo")
    axes[1].plot(step_counts, matched_prices, "s--", color="#b8862a", label="analytic (BGK-matched)")
    axes[1].axhline(continuous_price, color="#555555", linestyle=":", label="analytic (continuous)")
    axes[1].set_xlabel("monitoring steps")
    axes[1].set_title("Discrete-monitoring bias closing as steps grow")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


interact_manual(
    plot_convergence,
    spot=FloatSlider(value=90.0, min=60.0, max=125.0, step=1.0),
    strike=FloatSlider(value=100.0, min=60.0, max=140.0, step=1.0),
    barrier=FloatSlider(value=130.0, min=105.0, max=160.0, step=1.0),
    sigma=FloatSlider(value=0.25, min=0.05, max=0.6, step=0.01),
    time_to_expiry=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1),
    rate=FloatSlider(value=0.03, min=0.0, max=0.08, step=0.005),
);

## 4. Delta and gamma near the barrier

The point of the whole exercise: gamma is not bounded as spot approaches the
barrier, unlike a vanilla option. The finite-difference bump is shrunk as
`spot` nears `barrier` (`rel_bump` below) so the stencil can actually resolve
the kink instead of stepping over it — a fixed bump either misses the effect
entirely (too coarse, far from the barrier) or becomes numerically unstable
(too fine, extremely close to it).

In [ ]:
def plot_greeks(spot_min=70.0, spot_max=129.9, strike=100.0, barrier=130.0, sigma=0.25, time_to_expiry=1.0, rate=0.03, n_points=60):
    spots = np.linspace(spot_min, spot_max, n_points)
    prices, deltas, gammas = [], [], []
    for s in spots:
        base = dict(spot=s, strike=strike, barrier=barrier, time_to_expiry=time_to_expiry, rate=rate, sigma=sigma)
        dist_to_barrier = max(barrier - s, 1e-6)
        rel_bump = min(1e-3, (dist_to_barrier / barrier) * 0.3)
        prices.append(barrier_price_analytic(**base))
        deltas.append(barrier_delta(barrier_price_analytic, base, bump=rel_bump, relative=True))
        gammas.append(barrier_gamma(barrier_price_analytic, base, bump=rel_bump, relative=True))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, values, title in zip(axes, [prices, deltas, gammas], ["Price", "Delta", "Gamma"]):
        ax.plot(spots, values, color="#1f7a72")
        ax.axvline(barrier, color="#b8862a", linestyle="--", label="barrier")
        ax.set_xlabel("spot")
        ax.set_title(title)
    axes[0].legend()
    plt.tight_layout()
    plt.show()


interact_manual(
    plot_greeks,
    spot_min=FloatSlider(value=70.0, min=40.0, max=100.0, step=1.0),
    spot_max=FloatSlider(value=129.9, min=100.0, max=129.99, step=0.01),
    strike=FloatSlider(value=100.0, min=60.0, max=140.0, step=1.0),
    barrier=FloatSlider(value=130.0, min=105.0, max=160.0, step=1.0),
    sigma=FloatSlider(value=0.25, min=0.05, max=0.6, step=0.01),
    time_to_expiry=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1),
    rate=FloatSlider(value=0.03, min=0.0, max=0.08, step=0.005),
    n_points=IntSlider(value=60, min=20, max=150, step=10),
);